# Food Price Watch 01: data intake
Copies the 4 raw files this project uses from the old Drive folder (`MyDrive/FoodPriceWatch/data/raw/project`) into `MyDrive/FoodPriceWatch_v2/raw/`, then writes `raw/MANIFEST.csv`.
Files are only copied, never edited; an existing file is never overwritten. Run the cells top to bottom and paste the last cell's output back.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Code comes fresh from GitHub main each session; Drive holds data and outputs only.
import os
if not os.path.isdir('/content/foodprice_watch/.git'):
    !git clone --single-branch --branch main https://github.com/mrchowdhury587/foodprice_watch.git /content/foodprice_watch
else:
    !git -C /content/foodprice_watch pull --ff-only origin main
%pip install -q -r /content/foodprice_watch/requirements-colab.txt

In [ ]:
import sys, os, json
os.chdir('/content/foodprice_watch')
sys.path.insert(0, '/content/foodprice_watch/src')
CODE_COMMIT = !git rev-parse --short HEAD
CODE_COMMIT = CODE_COMMIT[0]
from foodwatch.paths import SETTINGS, DATA_ROOT, RAW, data, raw, sha256, write_csv
# stop if Drive is not really mounted (a failed mount leaves an empty /content/drive on the Colab machine)
assert os.path.ismount('/content/drive'), 'Google Drive is not mounted: Runtime > Disconnect and delete runtime, then rerun cell 1'
assert DATA_ROOT.is_dir(), f'{DATA_ROOT} not found in Google Drive (wrong account, or the folder is missing)'
for folder in ['raw', 'processed', 'reports/tables']:
    (DATA_ROOT / folder).mkdir(parents=True, exist_ok=True)
for tmp in DATA_ROOT.rglob('*.tmp'):   # half-written files from an interrupted session
    tmp.unlink()
print('code commit:', CODE_COMMIT, '| data root:', DATA_ROOT)

In [ ]:
import shutil
from pathlib import Path
OLD_RAW = Path(SETTINGS['drive']['old_raw'])
copy_log = []
for key, name in SETTINGS['raw_files'].items():
    src, dst = OLD_RAW / name, RAW / name
    if dst.exists():
        same = src.exists() and sha256(src) == sha256(dst)
        copy_log.append((name, 'already there' + ('' if same else ' (DIFFERS from old folder, kept as is)')))
    elif src.exists():
        shutil.copy2(src, dst); copy_log.append((name, 'copied'))
    else:
        copy_log.append((name, 'MISSING in old folder'))
for row in copy_log:
    print(*row, sep=': ')
assert all(raw(k).exists() for k in SETTINGS['raw_files']), 'a raw file is missing: put it into ' + str(RAW)

In [ ]:
import pandas as pd
from foodwatch import load
tables = {'prices': load.prices(), 'ramadan_eid': load.ramadan_eid(), 'fx': load.fx(), 'weather': load.weather()}
first_last = {'prices': ('Date',), 'fx': ('date',), 'weather': ('time',), 'ramadan_eid': ('ramadan_start',)}
rows = []
for key, df in tables.items():
    col = df[first_last[key][0]]
    rows.append(dict(file=SETTINGS['raw_files'][key], bytes=raw(key).stat().st_size, sha256=sha256(raw(key)), rows=len(df),
                     first=str(pd.to_datetime(col).min().date()), last=str(pd.to_datetime(col).max().date())))
manifest = pd.DataFrame(rows)
write_csv(manifest, data('manifest'))
item_ids = load.item_ids()
P = tables['prices']
found = {i: int((P.item_id == i).sum()) for i in item_ids}

In [ ]:
print('=== 01 data intake summary ===')
print('code commit:', CODE_COMMIT)
print(manifest.to_string(index=False))
print('price file: report days', P.Date.nunique(), '| items in file', P.item_id.nunique())
print('rows per focus item:', found)
print('CHECK all 8 items present:', all(n > 0 for n in found.values()))